# Modèle ARIMAX — Prédiction du Temps de Trajet Restant d’un Navire

Objectif : prédire **le temps restant avant l’arrivée d’un navire** à partir de sa **position actuelle (latitude, longitude)** et d’autres variables AIS.

Caractéristiques :
- Modèle **ARIMAX** (ARIMA avec variables exogènes)
- Entraînement sur **90% des trajets (tirés aléatoirement)**
- Test sur **10% restants**
- Possibilité d’obtenir **plusieurs temps restants probables** à partir de la distribution des résidus

Variables candidates :
- latitude
- longitude
- vitesse SOG
- variables météo (si disponibles)
- draft

Le modèle apprend la dynamique temporelle du **temps restant avant arrivée** le long d’un trajet.


## 0. Imports

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings("ignore")


## 1. Chargement des données AIS

In [2]:

DATA_PATH = '/home/onyxia/work/data/data_indian_ocean.feather'

print("Chargement des données...")
df = pd.read_feather(DATA_PATH)

df = df.sort_values("timestamp")

df.head()


Chargement des données...


,imo,mmsi,name,latitude,longitude,timestamp,sog,cog,nav status,nav status code,...,at port,port stay type,wave period Tp (s),significant wave height Hs (m),mean wave direction (°),sea surface temperature (°K),air temperature at 2m (°K),eastward wind velocity (m/s),northward wind velocity (m/s),mean wave direction relative to vessel (°)
5112191,9325697,311038000,AL AREESH,30.213982,32.557983,2010-01-01T01:38:39.000000,0.000000,90.000000,At Anchor,1,...,False,<NA>,NaN,NaN,NaN,NaN,285.391755,-1.088467,1.360746,NaN
4930862,9324435,240615000,AL JASSASIYA,15.671947,53.008030,2010-01-01T01:43:50.000000,20.573832,246.106262,Under Way Using Engine,0,...,False,<NA>,7.394453,0.762915,102.838568,298.466628,297.573985,-3.140716,-1.018815,143.267694
5202932,9325702,311133000,AL DAAYEN,30.213982,32.557983,2010-01-01T01:48:09.000000,0.000000,90.000000,Moored,5,...,False,<NA>,NaN,NaN,NaN,NaN,285.380828,-1.111017,1.360214,NaN
2493091,9250713,215533000,DISHA,26.300000,51.600000,2010-01-01T01:54:32.000000,0.000000,90.000000,At Anchor,1,...,False,<NA>,3.070412,0.319533,188.633052,295.298770,292.419544,4.814490,-1.196246,98.633052
3346810,9267015,310495000,LNG BENUE,-11.230967,98.037059,2010-01-01T01:57:08.000000,11.005275,237.908044,Under Way Using Engine,0,...,False,<NA>,8.652978,2.685963,133.335338,301.261785,299.003969,-10.655045,-0.399877,104.572706


## 2. Construction des trajets (trajectories)

In [3]:

# On suppose que chaque navire est identifié par MMSI

df["timestamp"] = pd.to_datetime(df["timestamp"])

# Création d'un identifiant de trajet simple
df = df.sort_values(["mmsi", "timestamp"])

df["time_diff"] = df.groupby("mmsi")["timestamp"].diff().dt.total_seconds().fillna(0)

# Nouveau trajet si gap > 12h
df["new_trip"] = df["time_diff"] > 12*3600

df["trip_id"] = df.groupby("mmsi")["new_trip"].cumsum()

df["trajectory_id"] = df["mmsi"].astype(str) + "_" + df["trip_id"].astype(str)

print("Nombre de trajectoires :", df["trajectory_id"].nunique())


Nombre de trajectoires : 25944


## 3. Calcul du temps restant avant arrivée

In [ ]:

trajectories = []

for traj_id, traj in df.groupby("trajectory_id"):

    traj = traj.sort_values("timestamp").copy()

    arrival_time = traj["timestamp"].iloc[-1]

    traj["remaining_time_sec"] = (arrival_time - traj["timestamp"]).dt.total_seconds()

    trajectories.append(traj)

df_traj = pd.concat(trajectories)

df_traj = df_traj[df_traj["remaining_time_sec"] > 0]

df_traj.head()

,imo,mmsi,name,latitude,longitude,timestamp,sog,cog,nav status,nav status code,...,sea surface temperature (°K),air temperature at 2m (°K),eastward wind velocity (m/s),northward wind velocity (m/s),mean wave direction relative to vessel (°),time_diff,new_trip,trip_id,trajectory_id,remaining_time_sec
2375224,9246621,205351000,EXCEL,30.311942,32.413102,2013-11-13 09:00:05,2.547450,157.676309,Under Way Using Engine,0,...,NaN,296.803490,-2.463714,-2.883919,NaN,0.0,False,0,205351000_0,986400.0
2375225,9246621,205351000,EXCEL,30.264149,32.458776,2013-11-13 10:00:05,7.504814,122.860929,Under Way Using Engine,0,...,NaN,299.142740,-2.664783,-3.262670,NaN,3600.0,False,0,205351000_0,982800.0
2375226,9246621,205351000,EXCEL,30.120157,32.516971,2013-11-13 11:00:05,9.158172,157.993663,Under Way Using Engine,0,...,NaN,299.411120,-2.120766,-3.314705,NaN,3600.0,False,0,205351000_0,979200.0
2375227,9246621,205351000,EXCEL,29.976166,32.575166,2013-11-13 12:00:05,8.734727,178.751782,Under Way Using Engine,0,...,NaN,299.410559,-1.414747,-3.322067,NaN,3600.0,False,0,205351000_0,975600.0
2375228,9246621,205351000,EXCEL,29.731890,32.693702,2013-11-13 13:00:05,17.016793,153.777458,Under Way Using Engine,0,...,NaN,299.300749,0.433916,-3.728501,NaN,3600.0,False,0,205351000_0,972000.0


## 4. Split Train / Test par Trajectoire (90 / 10)

In [5]:

unique_traj = df_traj["trajectory_id"].unique()

train_traj, test_traj = train_test_split(
    unique_traj,
    test_size=0.10,
    random_state=42
)

train_df = df_traj[df_traj["trajectory_id"].isin(train_traj)]
test_df = df_traj[df_traj["trajectory_id"].isin(test_traj)]

print("Trajets train :", len(train_traj))
print("Trajets test :", len(test_traj))


Trajets train : 23253
Trajets test : 2584


## 5. Variables du modèle ARIMAX

In [ ]:

TARGET = "remaining_time_sec"

EXOG_VARS = [
    "latitude",
    "longitude",
    #"sog",
    #"draft",
    #"wind_speed",
    #"significant wave height Hs (m)"
]

EXOG_VARS = [v for v in EXOG_VARS if v in train_df.columns]

print("Variables exogènes utilisées :", EXOG_VARS)

Variables exogènes utilisées : ['latitude', 'longitude']


## 6. Entraînement du modèle ARIMAX

In [ ]:
models = {}

for traj_id, traj in train_df.groupby("trajectory_id"):

    traj = traj.sort_values("timestamp")

    y = traj[TARGET]
    X = traj[EXOG_VARS] if EXOG_VARS else None

    if len(traj) < 20:
        continue

    model = SARIMAX(
        y,
        exog=X,
        order=(2,1,2),
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit(disp=False)

    models[traj_id] = results

print("Modèles entraînés :", len(models))

## 7. Évaluation sur les trajectoires test

In [ ]:

y_true = []
y_pred = []

for traj_id, traj in test_df.groupby("trajectory_id"):

    traj = traj.sort_values("timestamp")

    if len(traj) < 20:
        continue

    y = traj[TARGET]
    X = traj[EXOG_VARS] if EXOG_VARS else None

    model = SARIMAX(
        y,
        exog=X,
        order=(2,1,2),
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit(disp=False)

    pred = results.predict(start=0, end=len(y)-1, exog=X)

    y_true.extend(y)
    y_pred.extend(pred)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("MAE (secondes) :", mae)
print("RMSE (secondes) :", rmse)
print("MAE (heures) :", mae/3600)


NameError: name 'test_df' is not defined

## 8. Fonction d'interrogation du modèle

In [ ]:

def predict_remaining_time(model, lat, lon, sog=None, draft=None,
                           wind_speed=None, wave_height=None):

    X = pd.DataFrame({
        "latitude":[lat],
        "longitude":[lon],
        "sog":[sog],
        "draft":[draft],
        "wind_speed":[wind_speed],
        "significant wave height Hs (m)":[wave_height]
    })

    X = X[[c for c in EXOG_VARS if c in X.columns]]

    forecast = model.forecast(steps=1, exog=X)

    return float(forecast.iloc[0])


## 9. Générer plusieurs temps restants possibles (distribution)

In [ ]:

def predict_probabilistic(model, lat, lon, n_samples=3):

    base = predict_remaining_time(model, lat, lon)

    resid_std = np.std(model.resid)

    samples = np.random.normal(base, resid_std, n_samples)

    probs = np.ones(n_samples) / n_samples

    return pd.DataFrame({
        "remaining_time_sec": samples,
        "probability": probs
    })

## 10. Exemple d'utilisation

In [ ]:

example_model = list(models.values())[0]

predict_probabilistic(
    example_model,
    lat=12.5,
    lon=48.2
)
